# C2 — Learning-to-rank: pointwise vs pairwise vs listwise (H3)

Six gradient-boosted contenders that differ **only in the objective** (pointwise: XGBoost/LightGBM binary; pairwise: XGBoost `rank:pairwise` = RankNet loss; listwise: XGBoost and LightGBM LambdaMART, LightGBM XE-NDCG). Training queries = publication months; evaluation queries = B4's monthly batches. Each contender re-uses the exact C1 search draws of its library; all are selected on the v2 fold by **mean monthly validation NDCG@100**, refitted with 10 seeds and tested once. Design: Study C addendum 2 (commit 59bd4b1).

**H3:** best-validation LambdaMART beats the best-validation pointwise contender on per-month test NDCG@100 (month bootstrap CI of the mean difference > 0).

Reads saved outputs; full run (~25 min): `python -m src.models.c2_ltr --stage tune` then `--stage final`.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
from IPython.display import Image
pd.set_option("display.width", 250); pd.set_option("display.max_columns", 40)

## Validation search (v2 fold, 20 months)

In [2]:
t = pd.read_csv("outputs/c2_tuning.csv"); t.groupby("contender").val_ndcg_at_100.describe().round(4)

,count,mean,std,min,25%,50%,75%,max
contender,,,,,,,,
lgbm_binary,21.0,0.2285,0.0292,0.1694,0.2057,0.2262,0.2518,0.2740
lgbm_lambdamart,21.0,0.2465,0.0146,0.2200,0.2384,0.2497,0.2561,0.2726
lgbm_xendcg,21.0,0.0778,0.0375,0.0373,0.0457,0.0692,0.1022,0.1681
xgb_binary,21.0,0.2356,0.0218,0.1855,0.2166,0.2419,0.2505,0.2713
xgb_lambdamart,21.0,0.2463,0.0123,0.2129,0.2408,0.2472,0.2531,0.2656
xgb_pairwise,21.0,0.2493,0.0128,0.2278,0.2439,0.2479,0.2587,0.2832


In [3]:
pd.read_csv("outputs/c2_selected_configs.csv", index_col=0)[["config_id", "val_ndcg_at_100", "config"]]

,config_id,val_ndcg_at_100,config
contender,,,
lgbm_binary,15,0.273963,"{""n_estimators"": 232, ""learning_rate"": 0.02173..."
lgbm_lambdamart,19,0.272577,"{""n_estimators"": 612, ""learning_rate"": 0.01412..."
lgbm_xendcg,19,0.168093,"{""n_estimators"": 612, ""learning_rate"": 0.01412..."
xgb_binary,5,0.271326,"{""n_estimators"": 402, ""learning_rate"": 0.01662..."
xgb_lambdamart,7,0.265572,"{""n_estimators"": 534, ""learning_rate"": 0.03087..."
xgb_pairwise,18,0.283205,"{""n_estimators"": 529, ""learning_rate"": 0.01955..."


## Test results (10 seeds, mean ± std; global over all 226,463 test CVEs)

In [4]:
pd.read_csv("outputs/c2_summary.csv", index_col=0)

,pr_auc,precision_at_10,precision_at_100,ndcg_at_100,mrr,roc_auc
contender,,,,,,
xgb_pairwise,0.0194 ± 0.0007,0.22 ± 0.06,0.141 ± 0.0083,0.1507 ± 0.006,0.2258 ± 0.0637,0.8143 ± 0.0024
lgbm_lambdamart,0.0207 ± 0.0009,0.3 ± 0.0632,0.13 ± 0.01,0.1425 ± 0.0079,0.2136 ± 0.0415,0.8203 ± 0.0014
xgb_lambdamart,0.0205 ± 0.0014,0.26 ± 0.049,0.129 ± 0.0114,0.1415 ± 0.0118,0.2526 ± 0.0701,0.8279 ± 0.0024
lgbm_binary,0.0185 ± 0.0015,0.21 ± 0.07,0.123 ± 0.0149,0.1316 ± 0.0138,0.2187 ± 0.1157,0.833 ± 0.0034
xgb_binary,0.0179 ± 0.0008,0.15 ± 0.0671,0.124 ± 0.0111,0.1291 ± 0.0087,0.142 ± 0.0359,0.828 ± 0.0019
lgbm_xendcg,0.0114 ± 0.004,0.16 ± 0.1497,0.071 ± 0.0291,0.085 ± 0.0443,0.3323 ± 0.3511,0.6817 ± 0.0234


## H3 — LambdaMART vs pointwise, per-month NDCG@100 (46 months)

In [5]:
pd.read_csv("outputs/c2_h3.csv", index_col=0)

,value
lambdamart,lgbm_lambdamart
pointwise,lgbm_binary
mean_diff,0.00486570961221288
ci_low,-0.006069744453484701
ci_high,0.01583693785002618
p_not_better,0.2055
n_months,46
wilcoxon_statistic,502.5
wilcoxon_p_value,0.6779423228233218
wilcoxon_n,46


**H3 not supported.** The selected LambdaMART (LightGBM) is ahead of the selected pointwise model (LightGBM binary) by +0.0049 monthly NDCG@100, but the 95% month-bootstrap CI is [−0.0061, 0.0158] and the Wilcoxon median difference is 0 (p = 0.68). Ranking objectives did not measurably matter at this label density (~7 KEV per monthly queue).

## Secondary: global paired bootstrap (seed-averaged scores)

In [6]:
pd.read_csv("outputs/c2_secondary_tests.csv", index_col=0).round(4)

,pr_auc_diff,pr_auc_ci_low,pr_auc_ci_high,pr_auc_p_not_better,p_at_100_diff,p_at_100_ci_low,p_at_100_ci_high,p_at_100_p_not_better
lgbm_lambdamart_vs_lgbm_binary,0.0015,-0.0028,0.0073,0.224,-0.01,-0.04,0.03,0.738
xgb_pairwise_vs_lgbm_binary,-0.0007,-0.0058,0.0044,0.638,0.00,-0.04,0.04,0.568
lgbm_xendcg_vs_lgbm_binary,-0.0038,-0.0128,0.0030,0.854,-0.05,-0.10,0.00,0.988
best_lambdamart_vs_epss_pit,0.0037,-0.0020,0.0103,0.104,0.02,-0.03,0.08,0.264


## Friedman across the six objectives (46 months)

In [7]:
pd.read_csv("outputs/c2_friedman.csv", index_col=0).round(4)

,statistic,p_value,n_months,rank_xgb_pairwise,rank_xgb_lambdamart,rank_lgbm_binary,rank_lgbm_lambdamart,rank_xgb_binary,rank_lgbm_xendcg
ndcg_at_100,16.6297,0.0053,46.0,3.0326,3.1196,3.3043,3.5326,3.6739,4.3370
map,25.6646,0.0001,46.0,3.2609,3.1304,2.9783,3.4783,3.4348,4.7174


In [8]:
pd.read_csv("outputs/c2_nemenyi_ndcg_at_100.csv", index_col=0).round(3)

,lgbm_binary,lgbm_lambdamart,lgbm_xendcg,xgb_binary,xgb_lambdamart,xgb_pairwise
contender,,,,,,
lgbm_binary,1.000,0.992,0.086,0.934,0.997,0.982
lgbm_lambdamart,0.992,1.000,0.307,0.999,0.898,0.795
lgbm_xendcg,0.086,0.307,1.000,0.532,0.022,0.011
xgb_binary,0.934,0.999,0.532,1.000,0.714,0.569
xgb_lambdamart,0.997,0.898,0.022,0.714,1.000,1.000
xgb_pairwise,0.982,0.795,0.011,0.569,1.000,1.000


**Reading.** Friedman is significant (NDCG@100 p = 0.005, MAP p = 1e-4), driven mainly by XE-NDCG being clearly worst (it also had the most unstable validation search). Among the five working objectives the mean ranks are within ~0.6 of each other. Pairwise XGBoost has the highest global NDCG@100 and P@100, LambdaMART the highest global PR-AUC, but no pairwise difference to the pointwise model is significant. Nothing here beats point-in-time EPSS significantly (best LambdaMART − EPSS-PIT PR-AUC +0.0037 [−0.0020, 0.0103]).